# 🧭 트레이딩 에이전트 — Colab 퀵스타트

파이썬을 컴퓨터에 설치할 필요 없이, 이 페이지 안에서 바로 실행해볼 수 있습니다.
각 칸(셀) 왼쪽의 **▶ 재생 버튼**을 위에서 아래로 순서대로 누르기만 하면 됩니다.

> **안심하고 눌러보세요.** 이 노트북은 어떤 은행·증권 계좌에도 연결되어 있지 않습니다.
> 조건을 통과한 분석 결과는 **자동으로** 모의 계좌에 기록되지만, 그건 이 노트북 프로세스
> 안에서만 존재하는 가짜 장부입니다 — 실제 돈은 절대 움직이지 않습니다. 투자 자문이 아닌
> 연구·학습용 도구이고, 어떤 시스템도 수익을 보장할 수 없습니다.

## 1. 설치 (처음 한 번만 실행)

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/rlawntjd19/fantastic-fortnight.git"
REPO_DIR = "fantastic-fortnight"

if os.path.basename(os.getcwd()) != REPO_DIR:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir(REPO_DIR)

!pip install -q -r requirements.txt
print("설치 완료! 아래로 내려가서 다음 칸을 실행하세요.")

## 2. (선택) 진짜 AI 설명 문구 쓰기

이 칸을 건너뛰어도 프로그램은 완전히 정상 동작합니다 — 다만 각 애널리스트의 설명이
`[offline-stub] ...` 같은 임시 문구로 나옵니다. 실제 자연어 설명을 보고 싶다면:

1. Colab 왼쪽 사이드바의 **열쇠 모양 아이콘(Secrets)** 클릭
2. **이름**: `ANTHROPIC_API_KEY`, **값**: 본인의 API 키 입력 후 저장, 좌측의 토글을 켜서
   이 노트북에 접근을 허용
3. 아래 칸 실행

(신호/신뢰도/레버리지 같은 숫자는 API 키 유무와 관계없이 항상 코드로 직접 계산되므로,
이 단계를 건너뛰어도 결과의 신뢰성에는 차이가 없습니다.)

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("API 키를 불러왔습니다.")
except Exception:
    print("건너뛰었습니다 — 위 안내대로 Secrets에 키를 등록한 뒤 다시 실행하면 됩니다. (선택사항)")

## 3. 분석 + 자동 체결 + 자동 기록

아래 칸의 값을 원하는 대로 바꾼 뒤 실행하세요 (오른쪽의 입력창을 직접 클릭해서 수정 가능).
`leverage`를 아무리 크게 넣어도, 화면에는 안전 한도(기본 3배) 안으로 잘린 값이 나옵니다 —
의도된 동작입니다. 결과가 안전 기준을 통과하면 **별도 확인 없이 바로** 모의 계좌에 기록되고,
`trading_agent_journal.jsonl`에 판단 사유와 포트폴리오 변화가 자동으로 남습니다.

In [ ]:
symbol = "SK_HYNIX" #@param {type:"string"}
leverage = 5.0 #@param {type:"number"}
tranches = 2 #@param {type:"integer"}

!python -m trading_agent.cli signal "{symbol}" --leverage {leverage} --tranches {tranches}

## 4. (선택) 방금 기록된 내용 확인하기

위 3번 칸에서 기록된 거래의 사유와 그 결과 포트폴리오가 어떻게 바뀌었는지
`trading_agent_journal.jsonl`에서 바로 확인할 수 있습니다.

In [ ]:
import json, os

path = "trading_agent_journal.jsonl"
if os.path.exists(path):
    with open(path, encoding="utf-8") as f:
        lines = f.readlines()
    print(f"총 {len(lines)}건 기록됨. 가장 최근 기록:\n")
    print(json.dumps(json.loads(lines[-1]), indent=2, ensure_ascii=False))
else:
    print("아직 기록이 없습니다 — 위 3번 칸을 먼저 실행하세요.")

## 5. (선택) 실제 시세로 계속 반복 실행 + 대시보드

한 번만 분석하는 대신, 실제 Yahoo Finance 시세로 주기적으로 계속 분석하면서 자동으로
기록하고 싶다면 아래 칸을 실행하세요. `SK_HYNIX` 같은 임의의 이름이 아니라 진짜 티커를
넣어야 합니다 (SK하이닉스는 `000660.KS`, 애플은 `AAPL`). Colab에서는 `--dashboard`의
브라우저 접속이 되지 않으므로 대시보드 없이 터미널 로그로만 확인합니다. 중단하려면
이 칸의 ■ 정지 버튼을 누르세요.

In [ ]:
!pip install -q -r requirements-live.txt
live_symbol = "000660.KS" #@param {type:"string"}
interval_seconds = 30 #@param {type:"integer"}
max_ticks = 5 #@param {type:"integer"}

!python -m trading_agent.cli watch "{live_symbol}" --live --interval {interval_seconds} --max-iterations {max_ticks}

## 6. 브라우저 제어판(웹 UI) 열기

터미널 명령어 대신, 마우스로 클릭하는 대시보드로 쓰고 싶다면 아래 칸을 실행하세요.
종목 선택, 시작/일시정지/중단, 실시간 자산 곡선·포지션·판단 로그, 결과 내보내기(JSON/CSV/
Markdown/PDF)까지 전부 이 화면 안에서 할 수 있습니다. 아무것도 설치할 필요 없이 Colab
안에서 바로 열립니다.

In [ ]:
!pip install -q -r requirements-web.txt

import threading, time
import uvicorn

def _run_webapp():
    uvicorn.run("trading_agent.webapp.server:app", host="127.0.0.1", port=8000, log_level="warning")

threading.Thread(target=_run_webapp, daemon=True).start()
time.sleep(2)

from google.colab.output import serve_kernel_port_as_window
serve_kernel_port_as_window(8000)  # 새 창으로 열립니다 — 팝업 차단을 허용해주세요

---
연구·학습용 도구입니다. 투자 자문이 아니며, 실제 매매·자산 운용에 대한 책임을 지지 않습니다.
더 자세한 옵션과 설계 배경은 저장소의 `USAGE.md` / `README.md`를 참고하세요.